In [1]:
import json
import random
from tqdm import tqdm

INPUT_FILE = "phase1b_enron_train_reasoning.jsonl"
TRAIN_OUTPUT = "phase1b_enron_train_reasoning_filtered.jsonl"
TEST_OUTPUT = "phase1b_enron_test_reasoning_filtered.jsonl"

MAX_CHARS = 6000
MIN_CHARS = 50      

# Your exact requested distributions
TRAIN_NONE_TARGET = 800
TRAIN_ACTION_TARGET = 3200
TEST_NONE_TARGET = 200
TEST_ACTION_TARGET = 800

random.seed(42) 

print(f"Reading '{INPUT_FILE}' and stratifying into Action vs No-Action buckets...")

none_action_records = []
has_action_records = []
total_processed = 0

with open(INPUT_FILE, 'r', encoding='utf-8') as infile:
    for line in tqdm(infile, desc="Scanning Enron Emails"):
        total_processed += 1
        try:
            record = json.loads(line)
            email_text = str(record.get("input", ""))
            output_text = str(record.get("output", ""))
            
            if MIN_CHARS <= len(email_text) <= MAX_CHARS:
                # Group by presence of "Action: None"
                if "Action: None" in output_text:
                    none_action_records.append(line)
                else:
                    has_action_records.append(line)
        except json.JSONDecodeError:
            continue

print(f"\nFound {len(none_action_records):,} 'Action: None' emails.")
print(f"Found {len(has_action_records):,} 'Has Action' emails.")

print("\nSampling exact ratios for Training Set...")
# Use min() just in case the dataset has fewer records than requested
train_none = random.sample(none_action_records, min(TRAIN_NONE_TARGET, len(none_action_records)))
train_action = random.sample(has_action_records, min(TRAIN_ACTION_TARGET, len(has_action_records)))
train_records = train_none + train_action
random.shuffle(train_records) # Shuffle so the model doesn't memorize patterns

print("Removing used training records to prevent data leakage...")
# Convert to sets for fast removal
train_set_hashes = set(train_records)
remaining_none = [x for x in none_action_records if x not in train_set_hashes]
remaining_action = [x for x in has_action_records if x not in train_set_hashes]

print("Sampling exact ratios for Testing Set...")
test_none = random.sample(remaining_none, min(TEST_NONE_TARGET, len(remaining_none)))
test_action = random.sample(remaining_action, min(TEST_ACTION_TARGET, len(remaining_action)))
test_records = test_none + test_action
random.shuffle(test_records)

with open(TRAIN_OUTPUT, 'w', encoding='utf-8') as train_file:
    train_file.writelines(train_records)

with open(TEST_OUTPUT, 'w', encoding='utf-8') as test_file:
    test_file.writelines(test_records)

print("\n" + "="*50)
print("COMPLETED: STRATIFIED CLASS BALANCING")
print("="*50)
print(f"Training Set: {len(train_records):,} records saved to '{TRAIN_OUTPUT}'")
print(f" None: {len(train_none)} | Action: {len(train_action)}")
print(f"Testing Set:  {len(test_records):,} records saved to '{TEST_OUTPUT}'")
print(f" None: {len(test_none)} | Action: {len(test_action)}")
print("="*50)

Reading 'phase1b_enron_train_reasoning.jsonl' and stratifying into Action vs No-Action buckets...


Scanning Enron Emails: 209118it [00:02, 72248.01it/s]



Found 169,497 'Action: None' emails.
Found 27,276 'Has Action' emails.

Sampling exact ratios for Training Set...
Removing used training records to prevent data leakage...
Sampling exact ratios for Testing Set...

COMPLETED: STRATIFIED CLASS BALANCING
Training Set: 4,000 records saved to 'phase1b_enron_train_reasoning_filtered.jsonl'
 None: 800 | Action: 3200
Testing Set:  1,000 records saved to 'phase1b_enron_test_reasoning_filtered.jsonl'
 None: 200 | Action: 800
